# Experiments

We test SBGP against PySR and FFX baselines on a set of 10 problems from PMLB. This notebook is for analysis / making tables.

In [5]:

from sklearn.model_selection import train_test_split
from SBGP import SBGPRegressor, generate_bounds
from pysr import PySRRegressor
from ffx import FFXRegressor
import pandas as pd
import numpy as np
import random
import time
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)
import sklearn, pysr
print(sklearn.__version__)
print(pysr.__version__)



1.7.1
1.5.10


In [37]:
results_df = pd.read_csv("experiments_results_2026_05_11.csv")
results_df["X_bounds"] = results_df["X_bounds"].fillna("None")


# Check the number of rows is as expected

In [41]:
# 10 problems
# FFX deterministic
# PySR default 5 runs
# SBGP 3x3 parameter values and 5 runs and another 2 values and 5 runs
n_rows_expected = 10 + (10 * 5) + (10 * 5 * 3 * 3) + (10 * 5 * 2)
n_rows = len(results_df)
print(n_rows, n_rows_expected)

610 610


# Create a Latex table with results

Datasets as columns, models as rows

In [ ]:
def fmt_num(x):
    if pd.isna(x):
        return "---"
    if abs(x) >= 1000 or (0 < abs(x) < 0.01):
        return f"{x:.2g}"
    return f"{x:.2f}"

def fmt(mean, std=None):
    if std is None or pd.isna(std):
        return fmt_num(mean)
    return f"{fmt_num(mean)} ({fmt_num(std)})"


In [42]:
# Dataset ID legend (order from CSV)
datasets_ordered = list(dict.fromkeys(results_df["Dataset"]))
dataset_id = {name: i for i, name in enumerate(datasets_ordered)}

print("Dataset legend:")
for name, did in dataset_id.items():
    print(f"  {did}: {name}")

def make_transposed_table(df, metric):
    df = df.copy()
    df["DID"] = df["Dataset"].map(dataset_id)
    all_dids = list(range(len(datasets_ordered)))

    ffx_agg    = df[df["Model"] == "FFX"].groupby("DID")[metric].mean()
    pysr_agg   = df[df["Model"] == "PySR"].groupby("DID")[metric].agg(["mean", "std"])
    sbgp_agg = (df[df["Model"] == "SBGP"]
                    .groupby(["DID", "X_bounds", "initevals"])[metric]
                    .agg(["mean", "std"]))

    rows = []

    # FFX
    row = {"Model": "FFX"}
    for did in all_dids:
        row[did] = fmt(ffx_agg.get(did, float("nan")))
    rows.append(row)

    # PySR
    row = {"Model": "PySR"}
    for did in all_dids:
        if did in pysr_agg.index:
            row[did] = fmt(pysr_agg.loc[did, "mean"]) #, pysr_agg.loc[did, "std"])
        else:
            row[did] = "---"
    rows.append(row)

    # SBGP Experiment 1 (X-bounds and initevals)
    for xb in ['None', 'train', 'test']:
        for ie in [1000, 5000, 10000]:
            row = {"Model": f"SBGP ({xb}, {ie})"}
            for did in all_dids:
                key = (did, xb, ie)
                if key in sbgp_agg.index:
                    row[did] = fmt(sbgp_agg.loc[key, "mean"]) #, sbgp_agg.loc[key, "std"])
                else:
                    row[did] = "---"
            rows.append(row)

    # SBGP Experiment 2 (maxcohortlen)
    sbgp_agg = (df[df["Model"] == "SBGP"]
                    .groupby(["DID", "maxcohortlen"])[metric]
                    .agg(["mean", "std"]))

    for mc in [3, 5, 7]:
        row = {"Model": f"SBGP ({mc})"}
        for did in all_dids:
            key = (did, mc)
            if key in sbgp_agg.index:
                row[did] = fmt(sbgp_agg.loc[key, "mean"]) #, sbgp_agg.loc[key, "std"])
            else:
                row[did] = "---"
        rows.append(row)            

    table = pd.DataFrame(rows).set_index("Model")
    table.columns.name = None
    return table

for metric, label in [("Rsq_test", "Test $R^2$"), ("Rsq_train", "Train $R^2$"), ("Time", "Time (s)")]:
    t = make_transposed_table(results_df, metric)
    latex = t.to_latex(escape=False, caption=label, label=f"tab:{metric}")
    print(f"\n% ===== {label} =====")
    print(latex)
    open(f"experiments_results_transposed_{metric}_mean_exp2_combined.tex", "w").write(latex)


Dataset legend:
  0: 659_sleuth_ex1714
  1: 485_analcatdata_vehicle
  2: 1096_FacultySalaries
  3: 687_sleuth_ex1605
  4: 706_sleuth_case1202
  5: 210_cloud
  6: 678_visualizing_environmental
  7: 665_sleuth_case2002
  8: 230_machine_cpu
  9: 561_cpu

% ===== Test $R^2$ =====
\begin{table}
\caption{Test $R^2$}
\label{tab:Rsq_test}
\begin{tabular}{lllllllllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 \\
Model &  &  &  &  &  &  &  &  &  &  \\
\midrule
FFX & -2.91 & 0.43 & -4.40 & -2.70 & -246.74 & 0.82 & -36.48 & -6.23 & 0.42 & 0.97 \\
PySR & 0.78 & 0.57 & 0.90 & 0.29 & 0.73 & 0.59 & 0.42 & 0.13 & 0.89 & 0.96 \\
SBGP (None, 1000) & -2e+05 & 0.55 & 0.91 & 0.18 & 0.64 & 0.62 & -4e+05 & 0.19 & 0.75 & 0.93 \\
SBGP (None, 5000) & -2e+05 & 0.57 & 0.90 & 0.22 & 0.66 & 0.59 & -4e+05 & 0.18 & 0.74 & 0.94 \\
SBGP (None, 10000) & -4e+05 & 0.57 & 0.90 & 0.26 & 0.57 & 0.59 & -2e+05 & 0.19 & 0.72 & 0.93 \\
SBGP (train, 1000) & 0.80 & 0.53 & 0.93 & 0.25 & 0.63 & 0.60 & -2e+05 & 0.20 & 0.76 & 0

In [26]:
# Dataset ID legend (order from CSV)
datasets_ordered = list(dict.fromkeys(results_df["Dataset"]))
dataset_id = {name: i for i, name in enumerate(datasets_ordered)}

print("Dataset legend:")
for name, did in dataset_id.items():
    print(f"  {did}: {name}")

def make_transposed_table(df, metric):
    df = df.copy()
    df["DID"] = df["Dataset"].map(dataset_id)
    all_dids = list(range(len(datasets_ordered)))

    sbgp_agg = (df[df["Model"] == "SBGP"]
                    .groupby(["DID", "maxcohortlen"])[metric]
                    .agg(["mean", "std"]))

    rows = []

    for mc in [3, 5, 7]:
        row = {"Model": f"SBGP ({mc})"}
        for did in all_dids:
            key = (did, mc)
            if key in sbgp_agg.index:
                row[did] = fmt(sbgp_agg.loc[key, "mean"]) #, sbgp_agg.loc[key, "std"])
            else:
                row[did] = "---"
        rows.append(row)

    table = pd.DataFrame(rows).set_index("Model")
    table.columns.name = None
    return table

for metric, label in [("Rsq_test", "Test $R^2$"), ("Rsq_train", "Train $R^2$"), ("Time", "Time (s)")]:
    t = make_transposed_table(results_df, metric)
    latex = t.to_latex(escape=False, caption=label, label=f"tab:{metric}")
    print(f"\n% ===== {label} =====")
    print(latex)
    open(f"experiments_results_transposed_{metric}_mean_exp2.tex", "w").write(latex)


Dataset legend:
  0: 659_sleuth_ex1714
  1: 485_analcatdata_vehicle
  2: 1096_FacultySalaries
  3: 687_sleuth_ex1605
  4: 706_sleuth_case1202
  5: 210_cloud
  6: 678_visualizing_environmental
  7: 665_sleuth_case2002
  8: 230_machine_cpu
  9: 561_cpu

% ===== Test $R^2$ =====
\begin{table}
\caption{Test $R^2$}
\label{tab:Rsq_test}
\begin{tabular}{lllllllllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 \\
Model &  &  &  &  &  &  &  &  &  &  \\
\midrule
SBGP (3) & 0.76 & 0.59 & 0.91 & 0.23 & 0.63 & 0.61 & -2e+05 & 0.19 & 0.79 & 0.89 \\
SBGP (5) & 0.84 & 0.54 & 0.92 & 0.25 & 0.64 & 0.59 & -2e+05 & 0.18 & 0.78 & 0.94 \\
SBGP (7) & 0.80 & 0.54 & 0.92 & 0.25 & 0.64 & 0.62 & 0.21 & 0.19 & 0.77 & 0.95 \\
\bottomrule
\end{tabular}
\end{table}


% ===== Train $R^2$ =====
\begin{table}
\caption{Train $R^2$}
\label{tab:Rsq_train}
\begin{tabular}{lllllllllll}
\toprule
 & 0 & 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 \\
Model &  &  &  &  &  &  &  &  &  &  \\
\midrule
SBGP (3) & 0.68 & 0.68 & 0.82 & 0

# Stat test: compare a chosen SBGP model v PySR

In [44]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

df = results_df
# Filter to the two conditions
pysr = df[df["Model"] == "PySR"].copy()
sbgp = df[
    (df["Model"] != "PySR") &
    (df["X_bounds"] == "test") &
    (df["initevals"] == 5000) &
    (df["maxcohortlen"] == 7)
].copy()

print(f"PySR rows: {len(pysr)}, SBGP rows: {len(sbgp)}")
print(f"Unique datasets: {df['Dataset'].nunique()}")

subset = pd.concat([pysr, sbgp], ignore_index=True)

# --- Two-way ANOVA: Model + Dataset as blocking factor ---
lm = ols("Rsq_test ~ C(Model) + C(Dataset)", data=subset).fit()
table = sm.stats.anova_lm(lm, typ=2)
print("\nTwo-way ANOVA (Dataset as block):")
print(table.to_string())

# --- Paired test: average reps, then pair by dataset ---
means = subset.groupby(["Model", "Dataset"])["Rsq_test"].mean().unstack("Model")
t_stat, p_paired = stats.ttest_rel(means.iloc[:, 0], means.iloc[:, 1])
print(f"\nPaired t-test (mean over reps, paired by dataset):")
print(f"  {means.columns.tolist()[0]} vs {means.columns.tolist()[1]}")
print(f"  t={t_stat:.4f}, p={p_paired:.4f}")
print("\nPer-dataset means:")
print(means.to_string())


PySR rows: 50, SBGP rows: 50
Unique datasets: 10

Two-way ANOVA (Dataset as block):
            sum_sq    df     F  PR(>F)
C(Model)      0.03  1.00  3.86    0.05
C(Dataset)    7.04  9.00 97.10    0.00
Residual      0.72 89.00   NaN     NaN

Paired t-test (mean over reps, paired by dataset):
  SBGP vs PySR
  t=-1.3511, p=0.2097

Per-dataset means:
Model                          SBGP  PySR
Dataset                                    
1096_FacultySalaries             0.92  0.90
210_cloud                        0.62  0.59
230_machine_cpu                  0.77  0.89
485_analcatdata_vehicle          0.54  0.57
561_cpu                          0.95  0.96
659_sleuth_ex1714                0.80  0.78
665_sleuth_case2002              0.19  0.13
678_visualizing_environmental    0.21  0.42
687_sleuth_ex1605                0.25  0.29
706_sleuth_case1202              0.64  0.73


In [45]:
print("Mean R² test per method:")
print(means.mean().sort_values(ascending=False).to_string())
print(f"\nPySR wins on {(means['PySR'] > means['SBGP']).sum()}/10 datasets")
print(f"SBGP wins on {(means['SBGP'] > means['PySR']).sum()}/10 datasets")


Mean R² test per method:
Model
PySR     0.62
SBGP   0.59

PySR wins on 6/10 datasets
SBGP wins on 4/10 datasets
